In [ ]:
from __future__ import annotations
import numpy as np
import pandas as pd

import math
import random
import time
import warnings
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("--- Mac M4 GPU (MPS) Aktif!  ---")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("--- Nvidia GPU (CUDA) Aktif! ---")
else:
    device = torch.device("cpu")
    print("--- GPU Bulunamadı, CPU ile devam ediliyor. ---")

--- Mac M4 GPU (MPS) Aktif!  ---


In [18]:
df = pd.read_parquet("/Users/YGT/ist-airport-decision-support-system/data/gold/trajectories/trajectory_gold.parquet")

In [19]:
df.columns

Index(['flight_id', 'episode_id', 'point_timestamp', 'icao', 'date', 'e_m',
       'n_m', 'u_m', 'ux', 'uy', 'uz', 'r', 'sin_theta', 'cos_theta',
       'delta_t', 'distance_km', 'segment_id', 'gap_flag'],
      dtype='str')

In [ ]:
_SEG_OFFSET = 10_000

# ════════════════════════════════════════════════════════════════
# DEVICE
# ════════════════════════════════════════════════════════════════

def get_device() -> str:
    if torch.backends.mps.is_available():
        return "mps"
    if torch.cuda.is_available():
        return "cuda"
    return "cpu"


# ════════════════════════════════════════════════════════════════
# CONFIG
# ════════════════════════════════════════════════════════════════

@dataclass
class ATSCCConfig:
    # ── Paths
    traj_path: str = (
        "/Users/YGT/ist-airport-decision-support-system"
        "/data/gold/trajectories/trajectory_gold.parquet"
    )
    out_dir: str = (
        "/Users/YGT/ist-airport-decision-support-system"
        "/src/modeling/atscc_checkpoints"
    )

    # ── Veri
    n_flights:    Optional[int]    = None
    sample_seed:  int              = 42
    feature_cols: Tuple[str, ...] = (
        "e_m", "n_m", "u_m",
        "ux", "uy", "uz",
        "r", "sin_theta", "cos_theta",
        "delta_t",
        "gap_flag",
    )
    min_seq_len:  int    = 16
    max_seq_len:  int    = 256
    valid_ratio:  float  = 0.10
    flight_types: object = "all"

    # ── Model
    d_model:          int   = 256
    n_head:           int   = 8
    n_layer:          int   = 6
    d_ff:             int   = 1024
    d_out:            int   = 256
    dropout:          float = 0.40    # [v9-1]
    random_mask_prob: float = 0.30    # [v9-2]
    drop_path_rate:   float = 0.0

    # ── Training
    batch_size:              int   = 32
    accum_steps:             int   = 2
    num_workers:             int   = 0
    lr:                      float = 3e-6    # [v9-3]
    weight_decay:            float = 1e-2    # [v9-4]
    grad_clip:               float = 1.0
    epochs:                  int   = 60
    patience:                int   = 12
    min_delta:               float = 1e-4
    temperature:             float = 0.10
    max_timesteps_per_batch: int   = 8_000

    # ── LR Schedule
    warmup_epochs: int   = 5       # [v9-5]
    eta_min:       float = 1e-6    # [v9-6]

    # ── Runtime
    device:           str  = field(default_factory=get_device)
    save_best_only:   bool = True
    log_every:        int  = 1
    metric_token_cap: int  = 20_000


# ════════════════════════════════════════════════════════════════
# SEED
# ════════════════════════════════════════════════════════════════

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ════════════════════════════════════════════════════════════════
# FLIGHT TYPE DETECTION
# ════════════════════════════════════════════════════════════════

def classify_flight_type(g: pd.DataFrame) -> str:
    if "distance_km" not in g.columns:
        return "unknown"
    dist   = pd.to_numeric(g["distance_km"], errors="coerce").to_numpy(dtype=np.float32)
    finite = dist[np.isfinite(dist)]
    if len(finite) < 4:
        return "unknown"
    n          = max(1, len(finite) // 5)
    start_dist = float(np.nanmean(finite[:n]))
    end_dist   = float(np.nanmean(finite[-n:]))
    if not (np.isfinite(start_dist) and np.isfinite(end_dist)):
        return "unknown"
    delta = end_dist - start_dist
    if   delta < -5.0: return "arrival"
    elif delta >  5.0: return "departure"
    return "unknown"


# ════════════════════════════════════════════════════════════════
# LOCAL SEGMENT REMAP
# ════════════════════════════════════════════════════════════════

def local_segment_remap(seg: np.ndarray) -> np.ndarray:
    uniq    = pd.unique(seg)
    mapping = {int(old): int(new) for new, old in enumerate(uniq, start=1)}
    return np.array([mapping[int(s)] for s in seg], dtype=np.int64)


# ════════════════════════════════════════════════════════════════
# DATA LOADING
# ════════════════════════════════════════════════════════════════

def load_trajectories(cfg: ATSCCConfig) -> Dict[str, pd.DataFrame]:
    t0 = time.time()
    print("=" * 60)
    print("TRAJECTORY YÜKLENİYOR")
    print("=" * 60)

    traj = pd.read_parquet(cfg.traj_path)

    needed = set(cfg.feature_cols) | {"flight_id", "point_timestamp", "segment_id"}
    if cfg.flight_types != "all":
        needed |= {"distance_km"}

    missing = [c for c in needed if c not in traj.columns]
    if missing:
        raise ValueError(f"Eksik kolonlar: {missing}\nMevcut: {list(traj.columns)}")

    traj["flight_id"]       = traj["flight_id"].astype(str)
    traj["point_timestamp"] = pd.to_datetime(
        traj["point_timestamp"], utc=True, errors="coerce"
    )
    traj["segment_id"] = pd.to_numeric(traj["segment_id"], errors="coerce")
    for c in cfg.feature_cols:
        traj[c] = pd.to_numeric(traj[c], errors="coerce").astype("float32")
    if "distance_km" in traj.columns:
        traj["distance_km"] = pd.to_numeric(
            traj["distance_km"], errors="coerce"
        ).astype("float32")

    drop_cols = ["flight_id", "point_timestamp", "segment_id"] + list(cfg.feature_cols)
    traj      = traj.dropna(subset=drop_cols).copy()
    traj      = traj.sort_values(
        ["flight_id", "point_timestamp"], kind="mergesort"
    ).reset_index(drop=True)

    all_fids = traj["flight_id"].unique()
    print(f"  Toplam uçuş (raw)  : {len(all_fids):,}")
    print(f"  Toplam satır       : {len(traj):,}")

    if cfg.n_flights is not None:
        rng    = np.random.default_rng(cfg.sample_seed)
        chosen = rng.choice(all_fids, size=min(cfg.n_flights, len(all_fids)), replace=False)
        traj   = traj[traj["flight_id"].isin(chosen)].copy()
        print(f"  Alt küme seçildi   : {len(chosen):,}")

    flight_groups: Dict[str, pd.DataFrame] = {}
    skipped_short = 0
    skipped_type  = 0
    type_counts   = {"arrival": 0, "departure": 0, "unknown": 0}

    for fid, g in traj.groupby("flight_id", sort=False):
        g = g.sort_values("point_timestamp", kind="mergesort").reset_index(drop=True)

        if len(g) < cfg.min_seq_len:
            skipped_short += 1
            continue

        flight_type = classify_flight_type(g)
        type_counts[flight_type] += 1

        if cfg.flight_types != "all" and flight_type not in cfg.flight_types:
            skipped_type += 1
            continue

        if len(g) > cfg.max_seq_len:
            g = (
                g.iloc[:cfg.max_seq_len] if flight_type == "departure"
                else g.iloc[-cfg.max_seq_len:]
            ).reset_index(drop=True)

        g                = g.copy()
        g["segment_id"]  = local_segment_remap(g["segment_id"].to_numpy(dtype=np.int64))
        g["flight_type"] = flight_type
        flight_groups[str(fid)] = g

    print(f"  Tespit — arrival   : {type_counts['arrival']:,}")
    print(f"  Tespit — departure : {type_counts['departure']:,}")
    print(f"  Tespit — unknown   : {type_counts['unknown']:,}")
    print(f"  min_seq_len atılan : {skipped_short:,}")
    print(f"  tip filtresi atılan: {skipped_type:,}")
    print(f"  Kullanılabilir     : {len(flight_groups):,}")
    print(f"  Yükleme süresi     : {time.time() - t0:.1f}s")
    print()

    if len(flight_groups) == 0:
        raise RuntimeError("Hiç kullanılabilir uçuş bulunamadı!")
    return flight_groups


# ════════════════════════════════════════════════════════════════
# DATASET & COLLATE
# ════════════════════════════════════════════════════════════════

class ATSCCDataset(Dataset):
    def __init__(
        self,
        flight_groups: Dict[str, pd.DataFrame],
        flight_ids:    List[str],
        feature_cols:  Tuple[str, ...],
    ):
        self.flight_groups = flight_groups
        self.flight_ids    = flight_ids
        self.feature_cols  = feature_cols

    def __len__(self) -> int:
        return len(self.flight_ids)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        fid = self.flight_ids[idx]
        g   = self.flight_groups[fid]
        x   = g[list(self.feature_cols)].to_numpy(dtype=np.float32).copy()
        seg = g["segment_id"].to_numpy(dtype=np.int64).copy()
        return {
            "flight_id":  fid,
            "x":          torch.tensor(x,   dtype=torch.float32),
            "segment_id": torch.tensor(seg, dtype=torch.long),
        }


def collate_atscc(batch: List[Dict[str, Any]]) -> Dict[str, Any]:
    lengths  = [item["x"].shape[0] for item in batch]
    max_len  = max(lengths)
    feat_dim = batch[0]["x"].shape[1]
    B        = len(batch)

    x_pad            = torch.zeros((B, max_len, feat_dim), dtype=torch.float32)
    seg_pad          = torch.full((B, max_len), -1, dtype=torch.long)
    key_padding_mask = torch.ones((B, max_len), dtype=torch.bool)
    flight_ids: List[str] = []

    for i, item in enumerate(batch):
        T = item["x"].shape[0]
        x_pad[i, :T]            = item["x"]
        seg_pad[i, :T]          = item["segment_id"]
        key_padding_mask[i, :T] = False
        flight_ids.append(item["flight_id"])

    return {
        "flight_id":        flight_ids,
        "x":                x_pad,
        "segment_id":       seg_pad,
        "key_padding_mask": key_padding_mask,
        "lengths":          torch.tensor(lengths, dtype=torch.long),
    }


# ════════════════════════════════════════════════════════════════
# STOCHASTIC DEPTH
# ════════════════════════════════════════════════════════════════

class DropPath(nn.Module):
    def __init__(self, drop_prob: float = 0.0):
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if not self.training or self.drop_prob == 0.0:
            return x
        keep_prob = 1.0 - self.drop_prob
        keep      = (torch.rand(x.shape[0], 1, 1, device=x.device) < keep_prob).float()
        return x * keep / keep_prob


# ════════════════════════════════════════════════════════════════
# TRANSFORMER LAYER
# ════════════════════════════════════════════════════════════════

class TransformerLayerWithDropPath(nn.Module):
    def __init__(
        self,
        d_model:        int,
        n_head:         int,
        d_ff:           int,
        dropout:        float,
        drop_path_rate: float,
    ):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.attn  = nn.MultiheadAttention(
            embed_dim   = d_model,
            num_heads   = n_head,
            dropout     = dropout,
            batch_first = True,
        )
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )
        self.drop_path = DropPath(drop_path_rate)

    def forward(
        self,
        x:                torch.Tensor,
        attn_mask:        Optional[torch.Tensor] = None,
        key_padding_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        attn_out, _ = self.attn(
            self.norm1(x), self.norm1(x), self.norm1(x),
            attn_mask        = attn_mask,
            key_padding_mask = key_padding_mask,
            need_weights     = False,
        )
        x = x + self.drop_path(attn_out)
        x = x + self.drop_path(self.ff(self.norm2(x)))
        return x


# ════════════════════════════════════════════════════════════════
# MODEL
# ════════════════════════════════════════════════════════════════

class ATSCCEncoder(nn.Module):
    def __init__(
        self,
        d_input:          int,
        d_model:          int,
        n_head:           int,
        n_layer:          int,
        d_ff:             int,
        d_out:            int,
        dropout:          float,
        random_mask_prob: float,
        drop_path_rate:   float = 0.0,
    ):
        super().__init__()
        self.random_mask_prob = random_mask_prob
        self.drop_path_rate   = drop_path_rate

        self.in_proj = nn.Linear(d_input, d_model, bias=True)
        self.in_ln   = nn.LayerNorm(d_model)

        dpr = [float(r) for r in np.linspace(0, drop_path_rate, n_layer)]
        self.layers = nn.ModuleList([
            TransformerLayerWithDropPath(
                d_model        = d_model,
                n_head         = n_head,
                d_ff           = d_ff,
                dropout        = dropout,
                drop_path_rate = dpr[i],
            )
            for i in range(n_layer)
        ])
        self.final_ln = nn.LayerNorm(d_model)
        self.out_proj = nn.Linear(d_model, d_out, bias=True)
        self.out_ln   = nn.LayerNorm(d_out)

    def forward(
        self,
        x:                torch.Tensor,
        key_padding_mask: torch.Tensor,
    ) -> torch.Tensor:
        x = torch.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

        if self.training and self.random_mask_prob > 0:
            rand_mask   = (
                torch.rand(key_padding_mask.shape, device=x.device) < self.random_mask_prob
            ) & (~key_padding_mask)
            x           = x.masked_fill(rand_mask.unsqueeze(-1), 0.0)
            merged_mask = key_padding_mask | rand_mask
        else:
            merged_mask = key_padding_mask

        h = F.normalize(self.in_ln(self.in_proj(x)), p=2, dim=-1)

        T           = h.shape[1]
        causal_mask = torch.triu(
            torch.ones(T, T, device=h.device, dtype=torch.bool), diagonal=1
        )

        for layer in self.layers:
            h = layer(h, attn_mask=causal_mask, key_padding_mask=merged_mask)

        h = self.final_ln(h)
        z = F.normalize(self.out_ln(self.out_proj(h)), p=2, dim=-1)
        return z


# ════════════════════════════════════════════════════════════════
# UNIQUE SEGMENT KEY
# ════════════════════════════════════════════════════════════════

def _make_seg_uid(
    flight_flat: torch.Tensor,
    seg_flat:    torch.Tensor,
) -> torch.Tensor:
    return flight_flat * _SEG_OFFSET + seg_flat


# ════════════════════════════════════════════════════════════════
# LOSS
# ════════════════════════════════════════════════════════════════

def atscc_snn_loss(
    z:                torch.Tensor,
    seg:              torch.Tensor,
    key_padding_mask: torch.Tensor,
    temperature:      float,
    max_n:            int,
) -> torch.Tensor:
    valid       = ~key_padding_mask
    z_flat      = z[valid]
    seg_flat    = seg[valid]

    B, T        = seg.shape
    flight_idx  = torch.arange(B, device=z.device).unsqueeze(1).expand(B, T)
    flight_flat = flight_idx[valid]

    N = z_flat.size(0)
    if N > max_n:
        idx         = torch.randperm(N, device=z.device)[:max_n]
        z_flat      = z_flat[idx]
        seg_flat    = seg_flat[idx]
        flight_flat = flight_flat[idx]
        N           = max_n

    if N < 2:
        return z.new_tensor(0.0, requires_grad=True)

    seg_uid = _make_seg_uid(flight_flat, seg_flat)
    sim     = torch.matmul(z_flat, z_flat.T) / temperature
    sim     = sim - sim.max(dim=1, keepdim=True).values.detach()

    eye          = torch.eye(N, device=z.device, dtype=torch.bool)
    same_flight  = flight_flat.unsqueeze(0) == flight_flat.unsqueeze(1)
    same_seg_uid = seg_uid.unsqueeze(0)     == seg_uid.unsqueeze(1)

    pos_mask = same_flight & same_seg_uid & (~eye)
    neg_mask = (~same_seg_uid) & (~eye)

    exp_sim  = torch.exp(sim)
    pos_sum  = (exp_sim * pos_mask).sum(dim=1)
    neg_sum  = (exp_sim * neg_mask).sum(dim=1)

    valid_rows = pos_mask.sum(dim=1) > 0
    if valid_rows.sum() == 0:
        return z.new_tensor(0.0, requires_grad=True)

    loss = -torch.log(
        (pos_sum[valid_rows] + 1e-12) /
        (pos_sum[valid_rows] + neg_sum[valid_rows] + 1e-12)
    )
    return loss.mean()


# ════════════════════════════════════════════════════════════════
# METRICS
# ════════════════════════════════════════════════════════════════

@torch.no_grad()
def compute_metrics(
    z_flat:      torch.Tensor,
    seg_flat:    torch.Tensor,
    flight_flat: torch.Tensor,
) -> Tuple[float, float]:
    z   = F.normalize(z_flat, p=2, dim=-1)
    N   = len(z)
    if N < 2:
        return float("nan"), float("nan")

    seg_uid      = _make_seg_uid(flight_flat.to(z.device), seg_flat.to(z.device))
    eye          = torch.eye(N, device=z.device, dtype=torch.bool)
    same_flight  = flight_flat.unsqueeze(0) == flight_flat.unsqueeze(1)
    same_seg_uid = seg_uid.unsqueeze(0)     == seg_uid.unsqueeze(1)
    pos_mask     = same_flight & same_seg_uid & (~eye)

    sim        = torch.matmul(z, z.T)
    alignment  = sim[pos_mask].mean().item() if pos_mask.sum() > 0 else float("nan")
    sq_dist    = 2.0 - 2.0 * sim.clamp(-1.0, 1.0)
    uniformity = torch.log(torch.exp(-2.0 * sq_dist).mean() + 1e-12).item()
    return alignment, uniformity


# ════════════════════════════════════════════════════════════════
# LR SCHEDULER
# ════════════════════════════════════════════════════════════════

def build_scheduler(
    optimizer:     torch.optim.Optimizer,
    warmup_epochs: int,
    total_epochs:  int,
    eta_min:       float,
) -> torch.optim.lr_scheduler.LambdaLR:
    base_lr   = optimizer.param_groups[0]["lr"]
    min_ratio = eta_min / base_lr

    def lr_lambda(epoch: int) -> float:
        if epoch < warmup_epochs:
            return float(epoch + 1) / float(max(1, warmup_epochs))
        progress = (epoch - warmup_epochs + 1) / float(max(1, total_epochs - warmup_epochs))
        progress = min(max(progress, 0.0), 1.0)
        cosine   = 0.5 * (1.0 + math.cos(math.pi * progress))
        return min_ratio + (1.0 - min_ratio) * cosine

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)


# ════════════════════════════════════════════════════════════════
# TRAIN ONE EPOCH
# ════════════════════════════════════════════════════════════════

def train_one_epoch(
    model:     ATSCCEncoder,
    loader:    DataLoader,
    optimizer: torch.optim.Optimizer,
    cfg:       ATSCCConfig,
) -> float:
    model.train()
    total_loss   = 0.0
    n_steps      = 0
    pending_grad = False

    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(loader):
        x   = batch["x"].to(cfg.device)
        seg = batch["segment_id"].to(cfg.device)
        kpm = batch["key_padding_mask"].to(cfg.device)

        if cfg.device == "cuda":
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                z    = model(x, kpm)
                loss = atscc_snn_loss(z, seg, kpm, cfg.temperature, cfg.max_timesteps_per_batch)
        else:
            z    = model(x, kpm)
            loss = atscc_snn_loss(z, seg, kpm, cfg.temperature, cfg.max_timesteps_per_batch)

        (loss / cfg.accum_steps).backward()
        pending_grad  = True
        total_loss   += float(loss.item())
        n_steps      += 1

        if (step + 1) % cfg.accum_steps == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            pending_grad = False

    if pending_grad:
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)

    if cfg.device == "mps":
        torch.mps.empty_cache()

    return total_loss / max(n_steps, 1)


# ════════════════════════════════════════════════════════════════
# EVALUATE EPOCH
# ════════════════════════════════════════════════════════════════

@torch.no_grad()
def evaluate_epoch(
    model:  ATSCCEncoder,
    loader: DataLoader,
    cfg:    ATSCCConfig,
) -> Tuple[float, float, float]:
    model.eval()
    total_loss = 0.0
    n_steps    = 0

    all_z:    List[torch.Tensor] = []
    all_seg:  List[torch.Tensor] = []
    all_fl:   List[torch.Tensor] = []
    collected = 0
    cap       = cfg.metric_token_cap

    for batch in loader:
        x   = batch["x"].to(cfg.device)
        seg = batch["segment_id"].to(cfg.device)
        kpm = batch["key_padding_mask"].to(cfg.device)

        z    = model(x, kpm)
        loss = atscc_snn_loss(z, seg, kpm, cfg.temperature, cfg.max_timesteps_per_batch)
        total_loss += float(loss.item())
        n_steps    += 1

        if collected < cap:
            valid = ~kpm
            B, T  = seg.shape
            fl    = torch.arange(B, device=z.device).unsqueeze(1).expand(B, T)

            z_v   = z[valid].cpu()
            seg_v = seg[valid].cpu()
            fl_v  = fl[valid].cpu()

            remaining = cap - collected
            if z_v.shape[0] > remaining:
                z_v   = z_v[:remaining]
                seg_v = seg_v[:remaining]
                fl_v  = fl_v[:remaining]

            all_z.append(z_v)
            all_seg.append(seg_v)
            all_fl.append(fl_v)
            collected += z_v.shape[0]

    if cfg.device == "mps":
        torch.mps.empty_cache()

    val_loss = total_loss / max(n_steps, 1)

    if collected == 0:
        return val_loss, float("nan"), float("nan")

    alignment, uniformity = compute_metrics(
        torch.cat(all_z,   dim=0),
        torch.cat(all_seg, dim=0),
        torch.cat(all_fl,  dim=0),
    )
    return val_loss, alignment, uniformity


# ════════════════════════════════════════════════════════════════
# CHECKPOINT
# ════════════════════════════════════════════════════════════════

def save_checkpoint(
    path:      Path,
    model:     ATSCCEncoder,
    optimizer: torch.optim.Optimizer,
    scheduler: Any,
    epoch:     int,
    val_loss:  float,
    cfg:       ATSCCConfig,
) -> None:
    torch.save({
        "epoch":                epoch,
        "model_state_dict":     model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
        "val_loss":             val_loss,
        "config":               asdict(cfg),
    }, path)


# ════════════════════════════════════════════════════════════════
# MAIN TRAINING LOOP
# ════════════════════════════════════════════════════════════════

def train(cfg: ATSCCConfig) -> pd.DataFrame:
    seed_everything(cfg.sample_seed)

    out_dir   = Path(cfg.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    eff_batch = cfg.batch_size * cfg.accum_steps
    print("\n" + "=" * 60)
    print("ATSCC EĞİTİMİ — IST Airport  (v9_final)")
    print("=" * 60)
    print(f"  device          : {cfg.device}")
    print(f"  features        : {len(cfg.feature_cols)}D")
    print(f"  model           : d_model={cfg.d_model}, n_head={cfg.n_head}, "
          f"n_layer={cfg.n_layer}, d_out={cfg.d_out}")
    print(f"  dropout         : {cfg.dropout}  mask_prob={cfg.random_mask_prob}  "
          f"drop_path={cfg.drop_path_rate}")
    print(f"  effective batch : {cfg.batch_size} × {cfg.accum_steps} = {eff_batch}")
    print(f"  temperature     : {cfg.temperature}")
    print(f"  lr              : {cfg.lr}  wd={cfg.weight_decay}")
    print(f"  epochs          : {cfg.epochs}  warmup={cfg.warmup_epochs}  "
          f"patience={cfg.patience}")
    print(f"  flight_types    : {cfg.flight_types}")
    print("=" * 60 + "\n")

    flight_groups = load_trajectories(cfg)
    all_ids       = np.array(sorted(flight_groups.keys()))
    rng           = np.random.default_rng(cfg.sample_seed)
    rng.shuffle(all_ids)

    n_val     = max(1, int(len(all_ids) * cfg.valid_ratio))
    val_ids   = all_ids[:n_val].tolist()
    train_ids = all_ids[n_val:].tolist()
    print(f"Train: {len(train_ids):,}  |  Val: {len(val_ids):,}\n")

    pin = (cfg.device == "cuda")

    train_loader = DataLoader(
        ATSCCDataset(flight_groups, train_ids, cfg.feature_cols),
        batch_size  = cfg.batch_size,
        shuffle     = True,
        num_workers = cfg.num_workers,
        collate_fn  = collate_atscc,
        drop_last   = True,
        pin_memory  = pin,
    )
    val_loader = DataLoader(
        ATSCCDataset(flight_groups, val_ids, cfg.feature_cols),
        batch_size  = cfg.batch_size,
        shuffle     = False,
        num_workers = cfg.num_workers,
        collate_fn  = collate_atscc,
        drop_last   = False,
        pin_memory  = pin,
    )

    model = ATSCCEncoder(
        d_input          = len(cfg.feature_cols),
        d_model          = cfg.d_model,
        n_head           = cfg.n_head,
        n_layer          = cfg.n_layer,
        d_ff             = cfg.d_ff,
        d_out            = cfg.d_out,
        dropout          = cfg.dropout,
        random_mask_prob = cfg.random_mask_prob,
        drop_path_rate   = cfg.drop_path_rate,
    ).to(cfg.device)

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Model parametresi: {n_params:,}  ({n_params / 1e6:.1f}M)\n")

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr           = cfg.lr,
        weight_decay = cfg.weight_decay,
        betas        = (0.9, 0.999),
        eps          = 1e-8,
    )

    scheduler = build_scheduler(
        optimizer     = optimizer,
        warmup_epochs = cfg.warmup_epochs,
        total_epochs  = cfg.epochs,
        eta_min       = cfg.eta_min,
    )

    best_val_loss  = float("inf")
    best_epoch     = 0
    patience_count = 0
    best_ckpt_path = out_dir / "atscc_best.pt"
    last_ckpt_path = out_dir / "atscc_last.pt"
    history: List[Dict[str, Any]] = []

    print(
        f"{'Epoch':>6} | {'TrainLoss':>10} | {'ValLoss':>9} | "
        f"{'Align':>7} | {'Unif':>7} | {'LR':>9} | {'Time':>6}"
    )
    print("-" * 72)

    for epoch in range(1, cfg.epochs + 1):
        t0         = time.time()
        current_lr = optimizer.param_groups[0]["lr"]

        train_loss                      = train_one_epoch(model, train_loader, optimizer, cfg)
        val_loss, alignment, uniformity = evaluate_epoch(model, val_loader, cfg)

        scheduler.step()

        elapsed = time.time() - t0

        history.append({
            "epoch":      epoch,
            "train_loss": train_loss,
            "val_loss":   val_loss,
            "alignment":  alignment,
            "uniformity": uniformity,
            "lr":         current_lr,
        })

        if epoch % cfg.log_every == 0:
            align_str = f"{alignment:7.4f}" if np.isfinite(alignment)  else "    NaN"
            unif_str  = f"{uniformity:7.4f}" if np.isfinite(uniformity) else "    NaN"
            print(
                f"{epoch:>6} | {train_loss:>10.6f} | {val_loss:>9.6f} | "
                f"{align_str} | {unif_str} | {current_lr:>9.2e} | {elapsed:>5.1f}s"
            )

        save_checkpoint(last_ckpt_path, model, optimizer, scheduler, epoch, val_loss, cfg)

        if val_loss < (best_val_loss - cfg.min_delta):
            best_val_loss  = val_loss
            best_epoch     = epoch
            patience_count = 0
            save_checkpoint(best_ckpt_path, model, optimizer, scheduler, epoch, val_loss, cfg)
            print(f"  ✓ Best checkpoint  → {best_ckpt_path.name}  (val_loss={val_loss:.6f})")
        else:
            patience_count += 1
            if patience_count >= cfg.patience:
                print(
                    f"\n  Early stopping! "
                    f"Best epoch={best_epoch}, val_loss={best_val_loss:.6f}"
                )
                break

    print("\n" + "=" * 60)
    print("EĞİTİM TAMAMLANDI")
    print(f"  Best epoch    : {best_epoch}")
    print(f"  Best val_loss : {best_val_loss:.6f}")
    print(f"  Checkpoint    : {best_ckpt_path}")
    print("=" * 60)

    history_df   = pd.DataFrame(history)
    history_path = out_dir / "training_history.csv"
    history_df.to_csv(history_path, index=False)
    print(f"  History       : {history_path}")

    return history_df


# ════════════════════════════════════════════════════════════════
# INFERENCE UTILS
# ════════════════════════════════════════════════════════════════

def load_model(
    checkpoint_path: str,
    device:          Optional[str] = None,
) -> Tuple[ATSCCEncoder, ATSCCConfig]:
    ckpt         = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    valid_fields = ATSCCConfig.__dataclass_fields__.keys()
    cfg          = ATSCCConfig(**{k: v for k, v in ckpt["config"].items()
                                  if k in valid_fields})
    if device is not None:
        cfg.device = device

    model = ATSCCEncoder(
        d_input          = len(cfg.feature_cols),
        d_model          = cfg.d_model,
        n_head           = cfg.n_head,
        n_layer          = cfg.n_layer,
        d_ff             = cfg.d_ff,
        d_out            = cfg.d_out,
        dropout          = cfg.dropout,
        random_mask_prob = cfg.random_mask_prob,
        drop_path_rate   = getattr(cfg, "drop_path_rate", 0.0),
    ).to(cfg.device)

    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    print(
        f"Model yüklendi — epoch={ckpt['epoch']}, "
        f"val_loss={ckpt['val_loss']:.6f}, device={cfg.device}"
    )
    return model, cfg


@torch.no_grad()
def extract_embeddings(
    model:         ATSCCEncoder,
    flight_groups: Dict[str, pd.DataFrame],
    flight_ids:    List[str],
    cfg:           ATSCCConfig,
    batch_size:    int = 64,
) -> pd.DataFrame:
    model.eval()
    ds = ATSCCDataset(flight_groups, flight_ids, cfg.feature_cols)
    dl = DataLoader(
        ds, batch_size=batch_size, shuffle=False,
        num_workers=0, collate_fn=collate_atscc, pin_memory=False,
    )
    rows = []
    for batch in dl:
        x       = batch["x"].to(cfg.device)
        kpm     = batch["key_padding_mask"].to(cfg.device)
        lengths = batch["lengths"].to(cfg.device)
        z       = model(x, kpm)
        last    = lengths - 1
        bidx    = torch.arange(z.size(0), device=cfg.device)
        emb     = z[bidx, last].cpu().numpy()
        for fid, e in zip(batch["flight_id"], emb):
            rows.append({"flight_id": fid, "emb": e})
    return pd.DataFrame(rows)

In [6]:
cfg = ATSCCConfig(
    traj_path    = "/Users/YGT/ist-airport-decision-support-system/data/gold/trajectories/trajectory_gold.parquet",
    out_dir      = "/Users/YGT/ist-airport-decision-support-system/src/modeling/atscc_checkpoints_v10_reg",
    n_flights        = 100_000,
    flight_types     = {"arrival", "departure"},
    max_seq_len      = 256,
    min_seq_len      = 16,

    d_model          = 192,      # 256 -> 192
    n_head           = 8,
    n_layer          = 4,        # 6 -> 4
    d_ff             = 768,      # 1024 -> 768
    d_out            = 256,

    dropout          = 0.25,     # 0.40 -> 0.25
    random_mask_prob = 0.15,     # 0.30 -> 0.15
    drop_path_rate   = 0.10,     # 0.0  -> 0.10

    lr               = 7e-6,     # 3e-6 -> 7e-6
    weight_decay     = 5e-4,     # 1e-2 -> 5e-4
    grad_clip        = 1.0,

    warmup_epochs    = 4,
    eta_min          = 1e-6,
    epochs           = 40,
    patience         = 8,

    batch_size       = 32,
    accum_steps      = 2,
    num_workers      = 0,

    temperature      = 0.10,
    max_timesteps_per_batch = 8_000,
    metric_token_cap = 20_000,
    min_delta        = 1e-4,
)
history = train(cfg)
print(history.tail(10).to_string(index=False))


ATSCC EĞİTİMİ — IST Airport  (v9_final)
  device          : mps
  features        : 11D
  model           : d_model=192, n_head=8, n_layer=4, d_out=256
  dropout         : 0.25  mask_prob=0.15  drop_path=0.1
  effective batch : 32 × 2 = 64
  temperature     : 0.1
  lr              : 7e-06  wd=0.0005
  epochs          : 40  warmup=4  patience=8
  flight_types    : {'departure', 'arrival'}

TRAJECTORY YÜKLENİYOR
  Toplam uçuş (raw)  : 566,791
  Toplam satır       : 28,425,192
  Alt küme seçildi   : 100,000
  Tespit — arrival   : 40,450
  Tespit — departure : 41,318
  Tespit — unknown   : 5,910
  min_seq_len atılan : 12,322
  tip filtresi atılan: 5,910
  Kullanılabilir     : 81,768
  Yükleme süresi     : 40.2s

Train: 73,592  |  Val: 8,176

Model parametresi: 1,832,448  (1.8M)

 Epoch |  TrainLoss |   ValLoss |   Align |    Unif |        LR |   Time
------------------------------------------------------------------------
     1 |   1.563708 |  1.288612 |  0.3037 | -2.8521 |  1.75e-06 | 4

In [4]:
import numpy as np
import pandas as pd
from pathlib import Path

# atscc_train.py içinden geliyor varsayımı:
# - ATSCCConfig
# - load_model
# - load_trajectories
# - extract_embeddings

CKPT_PATH = "/Users/YGT/ist-airport-decision-support-system/src/modeling/atscc_checkpoints_v10_reg/atscc_best.pt"

model, cfg = load_model(CKPT_PATH, device="mps")

# eğitimdekiyle aynı subset/split yeniden oluşsun
flight_groups = load_trajectories(cfg)

all_ids = np.array(sorted(flight_groups.keys()))
rng = np.random.default_rng(cfg.sample_seed)
rng.shuffle(all_ids)

n_val = max(1, int(len(all_ids) * cfg.valid_ratio))
val_ids = all_ids[:n_val].tolist()
train_ids = all_ids[n_val:].tolist()

print("Train flights:", len(train_ids))
print("Val flights  :", len(val_ids))

Model yüklendi — epoch=17, val_loss=0.604500, device=mps
TRAJECTORY YÜKLENİYOR
  Toplam uçuş (raw)  : 566,791
  Toplam satır       : 28,425,192
  Alt küme seçildi   : 100,000
  Tespit — arrival   : 40,450
  Tespit — departure : 41,318
  Tespit — unknown   : 5,910
  min_seq_len atılan : 12,322
  tip filtresi atılan: 5,910
  Kullanılabilir     : 81,768
  Yükleme süresi     : 41.7s

Train flights: 73592
Val flights  : 8176
